# Day 4 – Delivery Time & Tip Prediction

Objective: Build supervised regression models to predict delivery time or tip amount.

In [1]:
# Import Libraries
import pandas as pd
import numpy as np
import os

In [2]:
# Load Datasets
customers = pd.read_csv("../data/raw/customers.csv")
restaurants = pd.read_csv("../data/raw/restaurants.csv")
drivers = pd.read_csv("../data/raw/drivers.csv")
orders = pd.read_csv("../data/raw/orders.csv")
deliveries = pd.read_csv("../data/raw/deliveries.csv")
reviews = pd.read_csv("../data/raw/reviews.csv")
payments = pd.read_csv("../data/raw/payments.csv")

In [3]:
# Orders
print(orders.shape)
print(orders.head())

# Deliveries
print(deliveries.shape)
print(deliveries.head())

# Restaurants
print(restaurants.shape)
print(restaurants.head())

# Delivery time target 
print(deliveries["delivery_time_minutes"].describe())
print(
    "Missing delivery times:",
    deliveries["delivery_time_minutes"].isna().sum()
)

# Order status
print(orders["order_status"].value_counts())

(10000, 7)
  order_id customer_id restaurant_id      order_timestamp  order_amount  \
0   O06946       C0115         R0082  2026-01-01 00:28:42       1859.36   
1   O04917       C0643         R0138  2026-01-01 00:45:00       1988.25   
2   O03258       C0091         R0194  2026-01-01 00:57:48       1773.35   
3   O01863       C0791         R0050  2026-01-01 01:04:17        418.24   
4   O03737       C0432         R0042  2026-01-01 01:15:54       1006.99   

   number_of_items order_status  
0                7    Delivered  
1                7    Delivered  
2                4    Delivered  
3                2    Delivered  
4                2    Delivered  
(8481, 8)
  delivery_id order_id driver_id  delivery_distance_km  \
0    DEL00001   O06946     D0020                 19.26   
1    DEL00002   O04917     D0291                  7.15   
2    DEL00003   O03258     D0162                 15.22   
3    DEL00004   O01863     D0238                  6.98   
4    DEL00005   O03737     D0047  

In [4]:
delivered_orders = orders[
    orders["order_status"] == "Delivered"
]

print(
    "Delivered orders:",
    len(delivered_orders)
)

print(
    "Delivered orders with delivery record:",
    delivered_orders["order_id"].isin(
        deliveries["order_id"]
    ).sum()
)

Delivered orders: 8481
Delivered orders with delivery record: 8481


In [5]:
customer_features = pd.read_csv(
    "../data/processed/customer_features.csv"
)

print(customer_features.shape)
print(customer_features.columns.tolist())

(1000, 12)
['customer_id', 'total_orders', 'avg_order_value', 'total_spending', 'active_months', 'ordering_frequency', 'weekend_orders', 'late_night_orders', 'avg_rating_given', 'cluster', 'cluster_name', 'dbscan_cluster']


In [6]:
# Merge delivered orders with delivery information
df = orders.merge(
    deliveries,
    on="order_id",
    how="inner"
)

print("After merging orders and deliveries:", df.shape)

# Add restaurant information
df = df.merge(
    restaurants,
    on="restaurant_id",
    how="left"
)

print("After adding restaurant information:", df.shape)

# Add customer segmentation information
df = df.merge(
    customer_features[
        ["customer_id", "cluster", "cluster_name"]
    ],
    on="customer_id",
    how="left"
)

print("Final dataset shape:", df.shape)

df.head()

After merging orders and deliveries: (8481, 14)
After adding restaurant information: (8481, 18)
Final dataset shape: (8481, 20)


,order_id,customer_id,restaurant_id,order_timestamp,order_amount,number_of_items,order_status,delivery_id,driver_id,delivery_distance_km,preparation_time_minutes,delivery_time_minutes,traffic_condition,weather,restaurant_name,restaurant_category,restaurant_rating,city,cluster,cluster_name
0,O06946,C0115,R0082,2026-01-01 00:28:42,1859.36,7,Delivered,DEL00001,D0020,19.26,44,141,Low,Rainy,South Indian Corner,Fast Food,4.0,Hyderabad,1,Frequent High-Value Customers
1,O04917,C0643,R0138,2026-01-01 00:45:00,1988.25,7,Delivered,DEL00002,D0291,7.15,40,79,Low,Cloudy,Taco Town,Chinese,3.7,Chennai,0,Occasional Lower-Value Customers
2,O03258,C0091,R0194,2026-01-01 00:57:48,1773.35,4,Delivered,DEL00003,D0162,15.22,21,113,High,Cloudy,Taco Town,Chinese,3.3,Chennai,0,Occasional Lower-Value Customers
3,O01863,C0791,R0050,2026-01-01 01:04:17,418.24,2,Delivered,DEL00004,D0238,6.98,11,53,Low,Cloudy,South Indian Corner,Fast Food,3.9,Mumbai,1,Frequent High-Value Customers
4,O03737,C0432,R0042,2026-01-01 01:15:54,1006.99,2,Delivered,DEL00005,D0047,17.07,24,107,Low,Rainy,Chinese Wok,Indian,3.7,Hyderabad,0,Occasional Lower-Value Customers


In [7]:
print(df.info())

print("\nMissing values:")
print(df.isnull().sum())

<class 'pandas.DataFrame'>
RangeIndex: 8481 entries, 0 to 8480
Data columns (total 20 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   order_id                  8481 non-null   str    
 1   customer_id               8481 non-null   str    
 2   restaurant_id             8481 non-null   str    
 3   order_timestamp           8481 non-null   str    
 4   order_amount              8481 non-null   float64
 5   number_of_items           8481 non-null   int64  
 6   order_status              8481 non-null   str    
 7   delivery_id               8481 non-null   str    
 8   driver_id                 8481 non-null   str    
 9   delivery_distance_km      8481 non-null   float64
 10  preparation_time_minutes  8481 non-null   int64  
 11  delivery_time_minutes     8481 non-null   int64  
 12  traffic_condition         8481 non-null   str    
 13  weather                   8481 non-null   str    
 14  restaurant_name    

In [9]:
df[
    [
        "order_id",
        "customer_id",
        "restaurant_id",
        "order_amount",
        "number_of_items",
        "delivery_distance_km",
        "preparation_time_minutes",
        "delivery_time_minutes",
        "traffic_condition",
        "weather",
        "restaurant_rating",
        "restaurant_category",
        "cluster_name"
    ]
].head()

,order_id,customer_id,restaurant_id,order_amount,number_of_items,delivery_distance_km,preparation_time_minutes,delivery_time_minutes,traffic_condition,weather,restaurant_rating,restaurant_category,cluster_name
0,O06946,C0115,R0082,1859.36,7,19.26,44,141,Low,Rainy,4.0,Fast Food,Frequent High-Value Customers
1,O04917,C0643,R0138,1988.25,7,7.15,40,79,Low,Cloudy,3.7,Chinese,Occasional Lower-Value Customers
2,O03258,C0091,R0194,1773.35,4,15.22,21,113,High,Cloudy,3.3,Chinese,Occasional Lower-Value Customers
3,O01863,C0791,R0050,418.24,2,6.98,11,53,Low,Cloudy,3.9,Fast Food,Frequent High-Value Customers
4,O03737,C0432,R0042,1006.99,2,17.07,24,107,Low,Rainy,3.7,Indian,Occasional Lower-Value Customers


In [10]:
# Feature engineering

# Convert timestamp to datetime
df["order_timestamp"] = pd.to_datetime(df["order_timestamp"])

# Extract time-based features
df["order_hour"] = df["order_timestamp"].dt.hour
df["day_of_week"] = df["order_timestamp"].dt.dayofweek

# Weekend indicator
df["is_weekend"] = (
    df["day_of_week"] >= 5
).astype(int)

# Time of day
def get_time_of_day(hour):
    if 5 <= hour < 12:
        return "Morning"
    elif 12 <= hour < 17:
        return "Afternoon"
    elif 17 <= hour < 22:
        return "Evening"
    else:
        return "Night"

df["time_of_day"] = df["order_hour"].apply(get_time_of_day)

df[
    [
        "order_timestamp",
        "order_hour",
        "day_of_week",
        "is_weekend",
        "time_of_day"
    ]
].head()

,order_timestamp,order_hour,day_of_week,is_weekend,time_of_day
0,2026-01-01 00:28:42,0,3,0,Night
1,2026-01-01 00:45:00,0,3,0,Night
2,2026-01-01 00:57:48,0,3,0,Night
3,2026-01-01 01:04:17,1,3,0,Night
4,2026-01-01 01:15:54,1,3,0,Night


In [ ]:
# Save the feature-engineered dataset
from pathlib import Path

output_path = Path("data/processed/delivery_prediction_features.csv")

df.to_csv(output_path, index=False)

print(f"Saved successfully: {output_path}")
print("Dataset shape:", df.shape)

Saved successfully: data/processed/delivery_prediction_features.csv
Dataset shape: (8481, 24)
